In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv("odi_final_dataset.csv")

xgb_model = joblib.load("xgb_model.pkl")

feature_columns = joblib.load("feature_columns.pkl")

In [3]:
world_cup_teams = [
    "India",
    "Australia",
    "England",
    "Pakistan",
    "New Zealand",
    "South Africa",
    "Sri Lanka",
    "Bangladesh",
    "West Indies",
    "Afghanistan"
]

In [4]:
team_strength = {}

teams = pd.concat(
    [df["team1"], df["team2"]]
).unique()

for team in teams:

    matches = df[
        (df["team1"] == team) |
        (df["team2"] == team)
    ]

    wins = (matches["winner"] == team).sum()

    team_strength[team] = wins / len(matches)

In [5]:
rankings = pd.DataFrame(
    team_strength.items(),
    columns=["Team", "Strength"]
)

rankings.sort_values(
    "Strength",
    ascending=False
).head(10)

,Team,Strength
5,India,0.673469
0,New Zealand,0.627119
4,England,0.589595
2,South Africa,0.582822
10,Scotland,0.571429
3,Australia,0.570588
18,United States of America,0.557143
19,Nepal,0.518519
17,Oman,0.515152
7,Pakistan,0.512346


In [6]:
rankings.sort_values(
    "Strength",
    ascending=False
).reset_index(drop=True)

,Team,Strength
0,India,0.673469
1,New Zealand,0.627119
2,England,0.589595
3,South Africa,0.582822
4,Scotland,0.571429
5,Australia,0.570588
6,United States of America,0.557143
7,Nepal,0.518519
8,Oman,0.515152
9,Pakistan,0.512346


In [7]:
wc_rankings = rankings.sort_values(
    "Strength",
    ascending=False
).reset_index(drop=True)

wc_rankings["Rank"] = range(1, len(wc_rankings) + 1)

wc_rankings = wc_rankings[
    ["Rank", "Team", "Strength"]
]

wc_rankings

,Rank,Team,Strength
0,1,India,0.673469
1,2,New Zealand,0.627119
2,3,England,0.589595
3,4,South Africa,0.582822
4,5,Scotland,0.571429
5,6,Australia,0.570588
6,7,United States of America,0.557143
7,8,Nepal,0.518519
8,9,Oman,0.515152
9,10,Pakistan,0.512346


In [8]:
recent_form_score = {}

for team in world_cup_teams:

    matches = df[
        (df["team1"] == team) |
        (df["team2"] == team)
    ].tail(10)

    wins = (matches["winner"] == team).sum()

    recent_form_score[team] = wins / 10

In [9]:
recent_form_score

{'India': np.float64(0.5),
 'Australia': np.float64(0.5),
 'England': np.float64(0.4),
 'Pakistan': np.float64(0.6),
 'New Zealand': np.float64(0.7),
 'South Africa': np.float64(0.4),
 'Sri Lanka': np.float64(0.4),
 'Bangladesh': np.float64(0.7),
 'West Indies': np.float64(0.2),
 'Afghanistan': np.float64(0.0)}

In [10]:
wc_score = {}

for team in world_cup_teams:

    strength = team_strength.get(team, 0)

    form = recent_form_score.get(team, 0)

    score = (
        0.7 * strength +
        0.3 * form
    )

    wc_score[team] = score

In [11]:
wc_table = pd.DataFrame(
    wc_score.items(),
    columns=["Team", "WC_Score"]
)

wc_table = wc_table.sort_values(
    "WC_Score",
    ascending=False
).reset_index(drop=True)

wc_table["Rank"] = range(
    1,
    len(wc_table) + 1
)

wc_table = wc_table[
    ["Rank", "Team", "WC_Score"]
]

wc_table

,Rank,Team,WC_Score
0,1,New Zealand,0.648983
1,2,India,0.621429
2,3,Bangladesh,0.567554
3,4,Australia,0.549412
4,5,Pakistan,0.538642
5,6,England,0.532717
6,7,South Africa,0.527975
7,8,Sri Lanka,0.413296
8,9,West Indies,0.302138
9,10,Afghanistan,0.000000


In [12]:
total_score = wc_table["WC_Score"].sum()

In [13]:
wc_table["Win_Chance_%"] = (
    wc_table["WC_Score"] /
    total_score
) * 100

In [14]:
wc_table["Win_Chance_%"] = (
    wc_table["Win_Chance_%"]
    .round(2)
)

In [15]:
wc_table

,Rank,Team,WC_Score,Win_Chance_%
0,1,New Zealand,0.648983,13.80
1,2,India,0.621429,13.22
2,3,Bangladesh,0.567554,12.07
3,4,Australia,0.549412,11.68
4,5,Pakistan,0.538642,11.46
5,6,England,0.532717,11.33
6,7,South Africa,0.527975,11.23
7,8,Sri Lanka,0.413296,8.79
8,9,West Indies,0.302138,6.43
9,10,Afghanistan,0.000000,0.00


In [16]:
world_cup_teams = [
    "India",
    "Australia",
    "England",
    "Pakistan",
    "New Zealand",
    "South Africa",
    "Sri Lanka",
    "Bangladesh",
    "West Indies",
    "Afghanistan"
]

In [17]:
from itertools import combinations

matches = list(
    combinations(world_cup_teams, 2)
)

len(matches)

45

In [18]:
def simulate_match(team1, team2):

    strength1 = team_strength.get(team1, 0.5)
    strength2 = team_strength.get(team2, 0.5)

    prob_team1 = strength1 / (strength1 + strength2)

    winner = np.random.choice(
        [team1, team2],
        p=[prob_team1, 1 - prob_team1]
    )

    return winner

In [19]:
def simulate_world_cup():

    points = {
        team: 0
        for team in world_cup_teams
    }

    for team1, team2 in matches:

        winner = simulate_match(
            team1,
            team2
        )

        points[winner] += 2

    champion = max(
        points,
        key=points.get
    )

    return champion

In [20]:
winner_count = {
    team: 0
    for team in world_cup_teams
}

In [21]:
for _ in range(1000):

    champion = simulate_world_cup()

    winner_count[champion] += 1

In [22]:
wc_prob = pd.DataFrame(
    winner_count.items(),
    columns=["Team", "Titles"]
)

In [23]:
wc_prob["Win_Chance_%"] = (
    wc_prob["Titles"] / 1000
) * 100

In [24]:
wc_prob = wc_prob.sort_values(
    "Win_Chance_%",
    ascending=False
)

wc_prob

,Team,Titles,Win_Chance_%
0,India,247,24.7
1,Australia,160,16.0
2,England,139,13.9
4,New Zealand,135,13.5
5,South Africa,89,8.9
3,Pakistan,88,8.8
7,Bangladesh,48,4.8
9,Afghanistan,41,4.1
6,Sri Lanka,33,3.3
8,West Indies,20,2.0


In [25]:
print(type(xgb_model))

print(len(feature_columns))

print(feature_columns[:20])

<class 'xgboost.sklearn.XGBClassifier'>
293
['team1_recent_form', 'team2_recent_form', 'team1_h2h_wins', 'team2_h2h_wins', 'team1_strength', 'team2_strength', 'toss_match_win', 'team1_Bangladesh', 'team1_Canada', 'team1_England', 'team1_Hong Kong', 'team1_India', 'team1_Ireland', 'team1_Jersey', 'team1_Namibia', 'team1_Nepal', 'team1_Netherlands', 'team1_New Zealand', 'team1_Oman', 'team1_Pakistan']


In [26]:
import pandas as pd
import numpy as np

In [27]:
def create_match_features(team1, team2):

    X_pred = pd.DataFrame(
        np.zeros((1, len(feature_columns))),
        columns=feature_columns
    )

    # -------------------
    # Recent Form
    # -------------------

    team1_form = get_recent_form(team1)
    team2_form = get_recent_form(team2)

    X_pred["team1_recent_form"] = team1_form
    X_pred["team2_recent_form"] = team2_form

    # -------------------
    # Head-to-Head
    # -------------------

    team1_h2h, team2_h2h = get_h2h(
        team1,
        team2
    )

    X_pred["team1_h2h_wins"] = team1_h2h
    X_pred["team2_h2h_wins"] = team2_h2h

    # -------------------
    # Team Strength
    # -------------------

    X_pred["team1_strength"] = team_strength.get(
        team1,
        0.5
    )

    X_pred["team2_strength"] = team_strength.get(
        team2,
        0.5
    )

    # -------------------
    # Toss Impact
    # -------------------

    X_pred["toss_match_win"] = np.random.randint(
        0,
        2
    )

    # -------------------
    # Team1 One-Hot Encoding
    # -------------------

    team1_col = f"team1_{team1}"

    if team1_col in X_pred.columns:
        X_pred[team1_col] = 1

    # -------------------
    # Team2 One-Hot Encoding
    # -------------------

    team2_col = f"team2_{team2}"

    if team2_col in X_pred.columns:
        X_pred[team2_col] = 1

    return X_pred

In [28]:
test = create_match_features(
    "India",
    "Australia"
)

test.head()

NameError: name 'get_recent_form' is not defined

In [ ]:
def simulate_match_xgb(team1, team2):

    X_pred = create_match_features(
        team1,
        team2
    )

    probs = xgb_model.predict_proba(X_pred)[0]

    prob_team2 = float(probs[0])
    prob_team1 = float(probs[1])

    total = prob_team1 + prob_team2

    prob_team1 = prob_team1 / total
    prob_team2 = prob_team2 / total

    winner = np.random.choice(
        [team1, team2],
        p=[prob_team1, prob_team2]
    )

    return winner

In [ ]:
for i in range(10):
    print(
        simulate_match_xgb(
            "India",
            "Australia"
        )
    )

In [ ]:
winner_count = {
    team: 0
    for team in world_cup_teams
}

In [ ]:
for i in range(1000):

    champion = simulate_world_cup_xgb()

    winner_count[champion] += 1

In [ ]:
wc_prob = pd.DataFrame(
    winner_count.items(),
    columns=["Team", "Titles"]
)

In [ ]:
wc_prob["Win_Chance_%"] = (
    wc_prob["Titles"] / 1000
) * 100

In [ ]:
wc_prob = wc_prob.sort_values(
    "Win_Chance_%",
    ascending=False
).reset_index(drop=True)

In [ ]:
wc_prob["Rank"] = range(
    1,
    len(wc_prob) + 1
)

In [ ]:
wc_prob = wc_prob[
    ["Rank", "Team", "Titles", "Win_Chance_%"]
]

In [ ]:
wc_prob

In [ ]:
def simulate_world_cup_playoffs():

    points = {
        team: 0
        for team in world_cup_teams
    }

    # League Stage

    for team1, team2 in matches:

        winner = simulate_match_xgb(
            team1,
            team2
        )

        points[winner] += 2

    # Points Table

    standings = pd.DataFrame(
        points.items(),
        columns=["Team", "Points"]
    )

    standings = standings.sort_values(
        "Points",
        ascending=False
    ).reset_index(drop=True)

    # Top 4 Teams

    top4 = standings.head(4)["Team"].tolist()

    # Semi Final 1

    sf1_winner = simulate_match_xgb(
        top4[0],
        top4[3]
    )

    # Semi Final 2

    sf2_winner = simulate_match_xgb(
        top4[1],
        top4[2]
    )

    # Final

    champion = simulate_match_xgb(
        sf1_winner,
        sf2_winner
    )

    return champion

In [ ]:
simulate_world_cup_playoffs()

In [ ]:
winner_count = {
    team: 0
    for team in world_cup_teams
}

In [ ]:
for i in range(1000):

    champion = simulate_world_cup_playoffs()

    winner_count[champion] += 1

In [ ]:
wc_prob = pd.DataFrame(
    winner_count.items(),
    columns=["Team","Titles"]
)

wc_prob["Win_Chance_%"] = (
    wc_prob["Titles"]/1000
)*100

wc_prob = wc_prob.sort_values(
    "Win_Chance_%",
    ascending=False
).reset_index(drop=True)

wc_prob["Rank"] = range(
    1,
    len(wc_prob)+1
)

wc_prob